# Putative Genetic Load
Given that the tara iti reference genome could not be annotated with RNA sequencing, ensure a robust and conservative assessment of genetic load that is translatable across species comparisons, we limited load estimates to highly conserved BUSCO genes in the fairy tern species complex (*Sterna nereis* spp.) and kakī (*Himantopus novazealandiae*) for comparison.  

To start, we ran BUSCO v5.4.7 for the tara iti and kakī reference genome.  

In [ ]:
busco --in Katie_racon_ragtag_autosomes.fa --out busco/ --mode genome --lineage_dataset aves_odb10 --cpu 32
busco --in himNova-hic-scaff_autosomes.fa --out busco/ --mode genome --lineage_dataset aves_odb10 --cpu 32

We then concatenated single copy BUSCO sequences into species specific `GFF` files.  

In [ ]:
cat busco/run_aves_odb10/busco_sequences/single_copy_busco_sequences/*.gff > SIFT_DB/TI_annotation/merged_scBUSCOs.gff
cat busco/run_aves_odb10/busco_sequences/single_copy_busco_sequences/*.gff > SIFT_DB/KI_annotation/merged_scBUSCOs.gff

cat busco/run_aves_odb10/busco_sequences/single_copy_busco_sequences/*.faa > SIFT_DB/TI_annotation/merged_scBUSCOs.fa
cat busco/run_aves_odb10/busco_sequences/single_copy_busco_sequences/*.faa > SIFT_DB/KI_annotation/merged_scBUSCOs.fa

First we fixed the concatenated `GFF` from BUSCO to be more compatible with Ensemble's Variant Effect Predictor (VEP) with  [agat](https://github.com/NBISweden/AGAT) v. 1.0.0.    

In [ ]:
agat_sp_manage_IDs.pl --gff KI_annotations/merged_scBUSCO.gff -o KI_annotations/merged_scBUSCO_checked.gff
agat_sp_manage_IDs.pl --gff TI_annotations/merged_scBUSCO.gff -o TI_annotations/merged_scBUSCO_checked.gff

Regions where the end position is before the start position were corrected before sorting, file compression with `bgzip` and indexing with `tabix`.  130 records in the TI genome and 55 records in the kaki genome had ending positions that occurred before the start positions. These start/end positions were flipped in order to not crash VEP.  

In [ ]:
cat KI_annotations/merged_scBUSCO_checked.gff | \
    awk '{ if ($5>$4) print $0}' | \
    sort -k1,1 -k4,4n -k5,5n -t$'\t' | bgzip -c > KI_annotations/merged_autosomal_checkedV2_noBadSites.gff.gz
cat TI_annotations/merged_scBUSCO_checked.gff | \
    awk '{ if ($5>$4) print $0}' | \
    sort -k1,1 -k4,4n -k5,5n -t$'\t' | bgzip -c > TI_annotations/merged_autosomal_checkedV2_noBadSites.gff.gz

tabix -p gff KI_annotations/scBUSCO_coorinates_fixed.gff.gz
tabix -p gff TI_annotations/scBUSCO_coorinates_fixed.gff.gz

The sum of sc-BUSCO gene regions is 289,743,567bp in kakī and 227,780,539bp in tara iti. This corresponds to roughly 26% and 21% of the kakī and fairy tern genomes. 

### Sites Polarised with ANGSD
To estimate masked and realised load in each fairy tern population and kakī we used ANGSD to output the major minor alleles, the called genotype, the posterior probability of the called genotype and all possible genotypes (`-doGeno 31`) to a BCF file (`-doBcf 1`).  

In [ ]:
angsd -P 24 -b ${ANGSD}GLOBAL.list -ref ${TREF} -anc ${TANC} -out ${ANGSD}samtools/genotypes/GLOBAL_polarized \
        -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 1 -trim 0 -C 50 -baq 1 -skipTriallelic 1 \
        -minMapQ 20 -minQ 20 -minInd 56 -setMinDepth 545 -setMaxDepth 1038 -doCounts 1 \
        -doPost 1 -postCutoff 0.95 -doBcf 1 -GL 1 -doMajorMinor 5 -doMaf 1 -SNP_pval 1e-6 -doGeno 31 --ignore-RG 0

angsd -P 32 -b ${DIR}KI.list -ref ${KREF} -anc ${KANC} -out ${DIR}samtools/genotypes/KI_polarized \
        -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 1 -trim 0 -C 50 -baq 1 -skipTriallelic 1 \
        -minMapQ 20 -minQ 20 -minInd 24 -setMinDepth 700 -maxDepth 1200 -doCounts 1 \
        -doPost 1 -postCutoff 0.95 -doBcf 1 -GL 1 -doMajorMinor 5 -doMaf 1 -SNP_pval 1e-6 -doGeno 31 --ignore-RG 0

### Variant Effect Predictor
Ensemble's [Variant Effect Predictor](https://www.ensembl.org/info/docs/tools/vep/vep_formats.html#output) (VEP) is a tool for analyzing tool for inferring the coding consequences of sequence variation. VEP was run below using the GTF files constructed above.  

In [ ]:
vep -i ${CHR}_snp.vcf \
    --custom file=TI_annotations/merged_autosomal_scBUSCOs_checkedV2_noBadSites.gff.gz,short_name=TI_BUSCOs,format=gff,type=overlap \
    --fasta Katie_autosomes.fa \
    -o ${CHR}_SNP_vep.vcf \
    --vcf --buffer_size 1

vep --vcf -i ${CHR}_sv.vcf \
    --custom file=TI_annotations/merged_autosomal_scBUSCOs_checkedV2_noBadSites.gff.gz,short_name=TI_BUSCOs,format=gff,type=overlap \
    --fasta Katie_autosomes.fa \
    -o ${CHR}_SV_vep.vcf \
    --vcf --buffer_size 1

VEP identifies the type of mutation and assigns a level of 'severity' to the mutation. This ranges from high, moderate, low, and modifier. A summary of each is in the table below. 

| Consequence |     Definition     |
|:-----------:|:------------------:|
|     Low     | Assumed to be mostly harmless or unlikely to change protein behaviour.  | 
|   Moderate  | A non-disruptive variant that might change protein effectiveness. |
|     High    | The variant is assumed to have high impact on the protein, probably causing truncation, loss of function, or triggering nonsense mediated decay. |
|   Modifier  | Usually non-coding variants or variants affecting non-codeing genes, where predictions are difficult or there is no evidence of impact. |

Given that the 'Modifier' consequence is assigned to mutations where predictions are difficult or there is a lack of evidence, the only mutations assigned as modifiers that were retained were intergenic variants. These are used later in our estimates of R<sub>*XY*</sub>.  

Below we count the number of alleles of each consequence for tara iti, Australian fairy tern. This loop was modified and run for kakī in the same manner.  

In [ ]:
# SNP genotypes
bcftools query -S AU.list -f '[%CHROM\t%POS\t%SAMPLE\t%GT\t%CSQ\n]' GLOBAL_SNP_vep.vcf \
    | sed 's/|/\t/g' | awk -f '{print $1"\t"$2"\t"$3"\t"$4"\t"$6"\t"$7"\tAU"}' > indiv_snp_vep_allele_freq.tsv
bcftools query -S TI.list -f '[%CHROM\t%POS\t%SAMPLE\t%GT\t%CSQ\n]' GLOBAL_SNP_vep.vcf \
    | sed 's/|/\t/g' | awk -f '{print $1"\t"$2"\t"$3"\t"$4"\t"$6"\t"$7"\tNZ"}' >> indiv_snp_vep_allele_freq.tsv
bcftools query -f '[%CHROM\t%POS\t%SAMPLE\t%GT\t%CSQ\n]' KI_variable_alt.vcf \
    | sed 's/|/\t/g' | awk -f '{print $1"\t"$2"\t"$3"\t"$4"\t"$6"\t"$7"\tKI"}' >> indiv_snp_vep_allele_freq.tsv

# SV genotypes - This was run on the VCF output from VEP to ensure the correct SV was correlated with the correct consequence
bcftools query -S AU.list -f '[%CHROM\t%POS\t%END\t%SVTYPE\t%SVLEN\t%SAMPLE\t%GT\t%CSQ\n]' Fairy_sv_vep.vcf \
    | sed 's/|/\t/g' | awk '{print $1"\t"$2"\t"$3"\t"$4"\t"$5"\t"$6"\t"$7"\t"$9"\t"$10"\tAU"}' > indiv_sv_vep_allele_freq.tsv
bcftools query -S TI.list -f '[%CHROM\t%POS\t%END\t%SVTYPE\t%SVLEN\t%SAMPLE\t%GT\t%CSQ\n]' Fairy_sv_vep.vcf \
    | sed 's/|/\t/g' | awk '{print $1"\t"$2"\t"$3"\t"$4"\t"$5"\t"$6"\t"$7"\t"$9"\t"$10"\tNZ"}' >> indiv_sv_vep_allele_freq.tsv
bcftools query -f '[%CHROM\t%POS\t%END\t%SVTYPE\t%SVLEN\t%SAMPLE\t%GT\t%CSQ\n]' KI_svs_VEP.vcf \
    | sed 's/|/\t/g' | awk '{print $1"\t"$2"\t"$3"\t"$4"\t"$5"\t"$6"\t"$7"\t"$9"\t"$10"\tKI"}' >> indiv_sv_vep_allele_freq.tsv

## Load Estimates
We first imported our python libraries for subsequent analyses.  


In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib.legend_handler import HandlerTuple
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import linregress
from sklearn.metrics import r2_score
from scipy.stats import mannwhitneyu
from scipy import stats

path = '/nesi/nobackup/uc03718/'
os.chdir(path)
print(os.getcwd())

/nesi/nobackup/uc03718


However, there were a few SNP sites in the fairy tern and kakī data sets that had ambiguous impacts. That is to say the same site had multiple mutation types and consequences. After loading the data from our files, these sites were excluded from all downstream analyses to ensure a conservative estimate of putative mutation load.  

In [ ]:
snp_vep = pd.read_csv('VEP/indiv_snp_vep_allele_freq.tsv', 
                      sep='\t', 
                      names=['Chromosome', 'Position', 'Sample', 'Genotype', 
                             'Mutation Class', 'Consequence', 'Population'])
sv_vep = pd.read_csv('VEP/indiv_sv_vep_allele_freq.tsv', 
                     sep='\t', 
                     names=['Chromosome', 'Start', 'End', 'SV Type', 'SV Length',
                            'Sample', 'Genotype', 'Mutation Class', 'Consequence',
                            'Population'])

C:\Users\jwold\AppData\Local\Temp\ipykernel_28804\266805531.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  indiv_vep['Allele Count'] = indiv_vep['Genotype'].replace(replace_map)


In [ ]:
# Now changing genotypes to allele counts
replace_map = {
    '0/0': 0,
    '0/1': 1,
    '1/1': 2
}

snp_vep['Allele Count'] = snp_vep['Genotype'].replace(replace_map)
snp_vep = snp_vep.drop(columns=['Genotype'])

sv_vep['Allele Count'] = sv_vep['Genotype'].replace(replace_map)
sv_vep = sv_vep.drop(columns=['Genotype'])

snp_conditions = [
    snp_vep['Allele Count'] == 0,
    snp_vep['Allele Count'] == 1,
    snp_vep['Allele Count'] == 2
]

sv_conditions = [
    sv_vep['Allele Count'] == 0,
    sv_vep['Allele Count'] == 1,
    sv_vep['Allele Count'] == 2
]

categories = ['Hom Ref', 'Het', 'Hom Alt']

snp_vep['Genotype Category'] = np.select(snp_conditions, categories, default='Unknown')
sv_vep['Genotype Category'] = np.select(sv_conditions, categories, default='Unknown')

In [ ]:
sv_vep.head()

Double checking that ambiguous sites have been excluded. 

In [ ]:
snpdupes = snp_vep[snp_vep.duplicated(subset=['Chromosome', 'Position',
                                              'Population', 'Sample'], keep=False)]
svdupes = sv_vep[sv_vep.duplicated(subset=['Chromosome', 'Start',
                                           'SV Type', 'Population', 'Sample'], keep=False)]

print("Number of duplicated SNP records: ", len(snpdupes), "Total number of SNP records: ", len(snp_vep))
print("Number of duplicated SV records: ", len(svdupes), "Total number of SV records: ", len(sv_vep))

0 32550306


We have 0 SNP dupes, so are good to progress to aggregating the individual counts into population counts. However, we found a few duplicates in the SVs. This is due to a potentially unresolved Duplication SV in the fairy tern population as observed below. 

In [ ]:
svdupes.to_csv('duplicated_SV_sites.tsv', sep='\t', index=True)
svdupes.drop_duplicates(subset=['Chromosome', 'Start', 'End', 'SV Type', 'SV Length']).head()

Because the SV type, mutation class, and consequence of this SV is the same, we excluded the larger SV from down stream analyses so as to not inflate our estimates of impactful SVs. This record was also excluded from estimates of heterozygosity, and population structure.  

In [ ]:
sv_vep = sv_vep[
    ~(
        (sv_vep['Chromosome']=='CM020452.1_RagTag') &
        (sv_vep['Start']==5295673) &
        (sv_vep['End']==5296087) &
        (sv_vep['SV Type']=='DUP')
    )
]

In [ ]:
pop_snps = snp_vep.groupby(['Chromosome', 'Position', 'Population'], as_index=False).agg({
    'Allele Count': 'sum',
    'Mutation Class': 'first',
    'Consequence': 'first'
})

pop_svs = sv_vep.groupby(['Chromosome', 'Start', 'End', 'SV Type', 'SV Length', 'Population'], as_index=False).agg({
    'Allele Count': 'sum',
    'Mutation Class': 'first',
    'Consequence': 'first'
})

# And now estimating allele frequency 
snp_conditions = [
    pop_snps['Population']=='AU',
    pop_snps['Population']=='NZ',
    pop_snps['Population']=='KI'
]

sv_conditions = [
    pop_svs['Population']=='AU',
    pop_svs['Population']=='NZ',
    pop_svs['Population']=='KI'
]

denominators = [2*19, 2*38, 2*24] # Double the population size for each

pop_snps['Allele Frequency'] = pop_snps['Allele Count'] / np.select(snp_conditions, denominators, default=np.nan)
pop_svs['Allele Frequency'] = pop_svs['Allele Count'] / np.select(sv_conditions, denominators, default=np.nan)
pop_snps.head()

In [ ]:
pop_svs.head()

### SNP SFS by mutation class and consequence
We used the SFS to explore the distribution of synonymous derived alleles in each population.  

In [ ]:
synonymous_count = pop_snps[pop_snps['Mutation Class']=='synonymous_variant']

au_counts = synonymous_count[synonymous_count['Population']=='AU'].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ti_counts = synonymous_count[synonymous_count['Population']=='NZ'].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ki_counts = synonymous_count[synonymous_count['Population']=='KI'].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")

plot_range = range(0, ti_counts['Allele Count'].max() + 1)

au_pivot = au_counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ti_pivot = ti_counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ki_pivot = ki_counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)

# Plot a stacked bar chart
au_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="gold")
plt.title("Frequency of Synonymous Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ti_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="steelblue")
plt.title("Frequency of Synonymous Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ki_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="black")
plt.title("Frequency of Synonymous Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

The highly binary share of the SFS for tara iti is going to be potentially problematic for later R<sub>XY</sub> estimates given the fact that the vast majority of these alleles are either absent or fixed, making the denominator == 0.  Will explore in more detail in the R<sub>XY</sub> section below.  

We also wanted to explore the shape of the SFS for missense muations and compare with the Low, Moderate, and High impact alleles as indicated by VEP.  

In [ ]:
missense_count = pop_snps[pop_snps['Mutation Class']=='missense_variant']

au_counts = missense_count[missense_count['Population']=='AU'].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ti_counts = missense_count[missense_count['Population']=='NZ'].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ki_counts = missense_count[missense_count['Population']=='KI'].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")

plot_range = range(77)

au_pivot = au_counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ti_pivot = ti_counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ki_pivot = ki_counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)

# Plot a stacked bar chart
au_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="gold")
plt.title("Frequency of Missense Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ti_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="steelblue")
plt.title("Frequency of Missense Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ki_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="black")
plt.title("Frequency of Missense Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

In [ ]:
au_LowCounts = pop_snps[(pop_snps['Population']=='AU') & (pop_snps['Consequence']=='LOW')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ti_LowCounts = pop_snps[(pop_snps['Population']=='NZ') & (pop_snps['Consequence']=='LOW')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ki_LowCounts = pop_snps[(pop_snps['Population']=='KI') & (pop_snps['Consequence']=='LOW')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")

plot_range = range(0, 77)

au_pivot = au_LowCounts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ti_pivot = ti_LowCounts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ki_pivot = ki_LowCounts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)

# Plot a stacked bar chart
au_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="gold")
plt.title("Frequency of Low Impact Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ti_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="steelblue")
plt.title("Frequency of Low Impact Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ki_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="black")
plt.title("Frequency of Low Impact Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

In [ ]:
au_Counts = pop_snps[(pop_snps['Population']=='AU') & (pop_snps['Consequence']=='HIGH')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ti_Counts = pop_snps[(pop_snps['Population']=='NZ') & (pop_snps['Consequence']=='HIGH')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ki_Counts = pop_snps[(pop_snps['Population']=='KI') & (pop_snps['Consequence']=='HIGH')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")

plot_range = range(0, 77)

au_pivot = au_Counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ti_pivot = ti_Counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ki_pivot = ki_Counts.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)

# Plot a stacked bar chart
au_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="gold")
plt.title("Frequency of High Impact Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ti_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="steelblue")
plt.title("Frequency of High Impact Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ki_pivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="black")
plt.title("Frequency of High Impact Allele Counts by Population")
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

Accross the board, we see that tara iti consistently have fewer derived alleles overall than either Australia fairy tern or kakī. In general, derived alleles are absent from the tara iti population in the Moderate or High impact categories, or either absent or fixed in the low category. This isn't surprising as the Low impact category is intended to capture nearly neutral alleles and as a result they are likely under less selection and more prone to genetic drift.  

### SV SFS by consequence
Unlike the SNP data where SNPs could be classified into Low, Moderate, or High impact, SVs generally were classified as either sequence modifiers (i.e., impact not well resolved) or High. This binary outcome of SV consequences is reflective of the difficulty in assessing the near and far sequence implications of SVs. 

In [ ]:
au_HighSVs = pop_svs[(pop_svs['Population']=='AU') & (pop_svs['Consequence']=='HIGH')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ti_HighSVs = pop_svs[(pop_svs['Population']=='NZ') & (pop_svs['Consequence']=='HIGH')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")
ki_HighSVs = pop_svs[(pop_svs['Population']=='KI') & (pop_svs['Consequence']=='HIGH')].groupby(["Allele Count", "Population"]).size().reset_index(name="Frequency")

plot_range = range(0, 77)

au_svpivot = au_HighSVs.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ti_svpivot = ti_HighSVs.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)
ki_svpivot = ki_HighSVs.pivot(index="Allele Count", columns="Population", values="Frequency").reindex(plot_range, fill_value=0)

au_svpivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="gold")
plt.title("Frequency of High Impact SV Allele Counts by Population")
#plt.xlim(-1,38)
plt.ylim(0,90)
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ti_svpivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="steelblue")
plt.title("Frequency of High Impact SV Allele Counts by Population")
#plt.xlim(-1,76)
plt.ylim(0,90)
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

ki_svpivot.plot(kind="bar", stacked=True, figsize=(15, 3), color="black")
plt.title("Frequency of High Impact SV Allele Counts by Population")
#plt.xlim(-1,24)
plt.ylim(0,90)
plt.ylabel("Frequency")
plt.xlabel("Allele Count")
plt.legend(title="Population", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

Immediately it is clear that Australian fairy tern have a high number of fixed High impact SVs. While this is likely true relative to the tara iti reference genome, it is challenging to know the context of these BUSCO regions in the AFT population of WA.  

In [ ]:
print('The mean allele frequency of putatively harmful SNPs and SVs in AFT: ', pop_snps[(pop_snps['Population']=='AU') & (pop_snps['Consequence']!='MODIFIER') & (pop_snps['Consequence']!='LOW')]['Allele Frequency'].mean(), ' & ', pop_svs[(pop_svs['Population']=='AU') & (pop_svs['Consequence']!='MODIFIER')]['Allele Frequency'].mean())
print('The mean allele frequency of putatively harmful SNPs and SVs in TI:  ', pop_snps[(pop_snps['Population']=='NZ') & (pop_snps['Consequence']!='MODIFIER') & (pop_snps['Consequence']!='LOW')]['Allele Frequency'].mean(), ' & ', pop_svs[(pop_svs['Population']=='NZ') & (pop_svs['Consequence']!='MODIFIER')]['Allele Frequency'].mean())
print('The mean allele frequency of putatively harmful SNPs and SVs in KI:  ', pop_snps[(pop_snps['Population']=='KI') & (pop_snps['Consequence']!='MODIFIER') & (pop_snps['Consequence']!='LOW')]['Allele Frequency'].mean(), ' & ', pop_svs[(pop_svs['Population']=='KI') & (pop_svs['Consequence']!='MODIFIER')]['Allele Frequency'].mean())

We then compared the number of derived sites present in each of the populations by their putative impact as ascertained by VEP.  

In [ ]:
plt.figure(figsize=(15,5))

nonzero_snp_counts = pop_snps[(pop_snps['Consequence']!='MODIFIER') & (pop_snps['Mutation Class']!='synonymous_variant') & (pop_snps['Allele Count']>0)].groupby(['Population', 'Consequence']).size().reset_index(name='Site Counts')
nonzero_sv_counts = pop_svs[(pop_svs['Consequence']!='MODIFIER') & (pop_svs['Allele Count']>0)].groupby(['Population', 'Consequence']).size().reset_index(name='Site Counts')

nonzero_snp_counts['Variant Type'] = 'SNP'
nonzero_sv_counts['Variant Type'] = 'SV' 

nonzero_counts = pd.concat([nonzero_snp_counts, nonzero_sv_counts])
nonzero_counts['group'] = nonzero_counts['Population'] + '-' + nonzero_counts['Variant Type']

sns.barplot(data=nonzero_counts, x='Consequence', y='Site Counts', 
            hue='group', hue_order=['AU-SNP', 'AU-SV', 'NZ-SNP', 'NZ-SV', 'KI-SNP', 'KI-SV'], 
            palette={'AU-SNP': 'gold', 'NZ-SNP': 'steelblue', 'KI-SNP':'black', 'AU-SV': 'orange', 'NZ-SV': 'lightblue', 'KI-SV': 'grey'}, 
            order=['LOW', 'MODERATE', 'HIGH'],
            dodge=True)
plt.title('Number of Derived Sites in Each Population')


We then estimated R<sub>xy</sub> between each of the three populations for putatively intolerant, tolerant, and all synonymous mutations. The [code](https://github.com/samarth8392/MQU_EvoGenomics/blob/main/RScripts/Rxy.R) below was adapted from [Mathur et al (2023)](http://doi.org/10.1093/evolut/qpac061).  

In [ ]:
def estimate_rxy(df, pop1, pop2, consequence, chroms_df):
    # Separate deleterious and synonymous data based on consequence type
    del_df = df[(df['Consequence'] == consequence) & (df['Mutation Class'] != 'synonymous_variant')]
    syn_df = df[df['Mutation Class'] == 'synonymous_variant']

    # Merge data for both populations
    del_merged = pd.merge(del_df[del_df['Population'] == pop1], del_df[del_df['Population'] == pop2],
                          on=['Chromosome', 'Position'], suffixes=('_pop1', '_pop2'))
    
    syn_merged = pd.merge(syn_df[syn_df['Population'] == pop1], syn_df[syn_df['Population'] == pop2],
                          on=['Chromosome', 'Position'], suffixes=('_pop1', '_pop2'))

    # Calculate l_xy and l_yx
    l_xy = np.sum(del_merged["Allele Frequency_pop1"] * (1 - del_merged["Allele Frequency_pop2"])) / np.sum(syn_merged["Allele Frequency_pop1"] * (1 - syn_merged["Allele Frequency_pop2"]))
    l_yx = np.sum(del_merged["Allele Frequency_pop2"] * (1 - del_merged["Allele Frequency_pop1"])) / np.sum(syn_merged["Allele Frequency_pop2"] * (1 - syn_merged["Allele Frequency_pop1"]))
    
    # Initialize results list
    Rxy_results = []

    # Loop over chromosomes
    for chr in chroms_df['Chromosome']:
        # Filter data excluding one chromosome at a time
        del_df_excluded = del_df[(del_df['Chromosome'] != chr)]
        syn_df_excluded = syn_df[(syn_df['Chromosome'] != chr)]

        # Merge excluded data for both populations
        del_excluded_merged = pd.merge(del_df_excluded[del_df_excluded['Population'] == pop1],
                                      del_df_excluded[del_df_excluded['Population'] == pop2],
                                      on=['Chromosome', 'Position'], suffixes=('_pop1', '_pop2'))
        
        syn_excluded_merged = pd.merge(syn_df_excluded[syn_df_excluded['Population'] == pop1],
                                      syn_df_excluded[syn_df_excluded['Population'] == pop2],
                                      on=['Chromosome', 'Position'], suffixes=('_pop1', '_pop2'))

        # Calculate l_xy_excluded and l_yx_excluded
        l_xy_excluded = np.sum(del_excluded_merged["Allele Frequency_pop1"] * (1 - del_excluded_merged["Allele Frequency_pop2"])) / np.sum(syn_excluded_merged["Allele Frequency_pop1"] * (1 - syn_excluded_merged["Allele Frequency_pop2"]))
        l_yx_excluded = np.sum(del_excluded_merged["Allele Frequency_pop2"] * (1 - del_excluded_merged["Allele Frequency_pop1"])) / np.sum(syn_excluded_merged["Allele Frequency_pop2"] * (1 - syn_excluded_merged["Allele Frequency_pop1"]))

        # Calculate rxy_excluded
        rxy_excluded = l_xy_excluded / l_yx_excluded

        # Append result to list
        Rxy_results.append({'Chromosome Excluded': chr, 'Rxy': rxy_excluded, 'Consequence': consequence})

    # Convert list of dictionaries to DataFrame
    Rxy_results_df = pd.DataFrame(Rxy_results)

    return Rxy_results_df

In [ ]:
pop_snps.head()

Filter the data to exclude KI from Rxy esimtates and those sites with ambiguous impacts (i.e., MODIFIER).  

In [ ]:
del_pop_allele = pop_snps[pop_snps['Population']!='KI']

grouped_del = del_pop_allele[(del_pop_allele['Consequence']=='MODERATE') | (del_pop_allele['Consequence']=='HIGH') | (del_pop_allele['Mutation Class']=='synonymous_variant')]
grouped_del['Consequence'] = 'grouped'

chromNames = del_pop_allele[del_pop_allele['Population']!='KI']
chromNames = chromNames['Chromosome'].sort_values().drop_duplicates()
chromNames = pd.DataFrame(chromNames, columns=['Chromosome'])

In [ ]:
grouped_del.head()

We then attempted to include SVs in our estimates of R<sub>XY</sub> by using our synonymous SNPs. For this we extracted synonymous SNPs and merged them with the SV data.  

In [ ]:
synonymous = pop_snps[(pop_snps['Mutation Class']=='synonymous_variant') & (pop_snps['Population']!='KI')]
synonymous['Variant Type'] = 'SNP'

sv_subset = pop_svs[['Chromosome', 'Start', 'SV Type', 'Population', 'Allele Count', 'Mutation Class', 'Consequence', 'Allele Frequency']]
sv_subset['Position'] = sv_subset['Start']
sv_subset['Variant Type'] = sv_subset['SV Type']
sv_subset = sv_subset.drop(columns=['Start', 'SV Type'])

sv_rxy_subset = pd.concat([synonymous, sv_subset], ignore_index=True)

sv_rxy_subset.tail()

In [ ]:
print("Mean Rxy of nearly neutral SNPs: ", low_Rxy['Rxy'].mean())
print("Mean Rxy of moderately deleterious SNPs: ", mod_Rxy['Rxy'].mean())
print("Mean Rxy of high impact SNPs: ", high_Rxy['Rxy'].mean())
print("Mean Rxy of high impact SVs: ", highsvRxy['Rxy'].mean())
print("Mean Rxy of SNPs with indeterminate impact: ", uncertain_snpRxy['Rxy'].mean())
print("Mean Rxy of SVs with indeterminate impact: ", uncertain_svRxy['Rxy'].mean())
print("Mean Rxy of grouped moderate and high impact SNPs ", grouped_Rxy['Rxy'].mean())

Sure enough, the SNPs in the Low impact category were too structured to reasonably compare relative estimates of Rxy in either population.  

In [ ]:
snpRxy = pd.concat([mod_Rxy, high_Rxy, grouped_Rxy, uncertain_snpRxy])
svRxy = pd.concat([highsvRxy, uncertain_svRxy])

fig, axes = plt.subplots(1, 2, figsize=(25, 8), sharex=False, sharey=False)

sns.violinplot(data=snpRxy, x='Consequence', y='Rxy', order=['MODIFIER', 'MODERATE', 'HIGH', 'grouped'], color='0.8', ax=axes[0])
sns.stripplot(data=snpRxy, x='Consequence', y='Rxy', order=['MODIFIER', 'MODERATE', 'HIGH', 'grouped'], jitter=True, color='#f4811d', dodge=True, ax=axes[0])
axes[0].axhline(y=1, color='black', linestyle='--', linewidth=0.75)
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_title('SNP R$_{XY}$')

sns.violinplot(data=svRxy, x='Consequence', y='Rxy', order=['MODIFIER', 'HIGH'], color='0.8', ax=axes[1])
sns.stripplot(data=svRxy, x='Consequence', y='Rxy', order=['MODIFIER', 'HIGH'], jitter=True, color='#f4811d', dodge=True, ax=axes[1])
axes[1].axhline(y=1, color='black', linestyle='--', linewidth=0.75)
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title('SV R$_{XY}$')

## Plots of putatively harmful allele counts and relative diversity
Now we examine the total number of putatively harmful derived alleles (`Allele Count`), then the number of putatively harmful derived alleles in a 'masked' state (`Het Allele Count`), and finally the number of puatively harmful alleles likely contributing to realised load (`Hom Allele Count`).

In [ ]:
snp_vep.head()

In [ ]:
snp_load = snp_vep[snp_vep['Consequence']!='MODIFIER']
snp_load['Pop-Consequence'] = snp_load['Population'] +'-' + snp_load['Consequence']

tot_snp = snp_load.groupby(['Sample', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
het_snp = snp_load[snp_load['Genotype Category']=='Het'].groupby(['Sample', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
alt_snp = snp_load[snp_load['Genotype Category']=='Hom Alt'].groupby(['Sample', 'Pop-Consequence'])['Allele Count'].sum().reset_index()

combinedRxy = pd.concat([mod_Rxy, high_Rxy])

palette = ['gold', 'steelblue', 'black']
order = ['AU-LOW', 'NZ-LOW', 'KI-LOW', 
         'AU-MODERATE', 'NZ-MODERATE', 'KI-MODERATE', 
         'AU-HIGH', 'NZ-HIGH', 'KI-HIGH']

fig, axes = plt.subplots(1, 4, figsize=(30, 8), sharex=False, sharey=False)

sns.violinplot(tot_snp, x='Pop-Consequence', y='Allele Count', order=order, color='0.8', ax=axes[0])
sns.stripplot(tot_snp, x='Pop-Consequence', y='Allele Count', hue='Pop-Consequence', hue_order=order, palette=palette, jitter=True, size=4, ax=axes[0])
axes[0].set_title('A)', loc='left', fontsize=20)
axes[0].set_xlabel('')
axes[0].set_ylabel('Individual Derived Allele Counts', fontsize=16)
axes[0].set_xticklabels(['Low', 'Low', 'Low', 'Moderate', 'Moderate', 'Moderate', 'High', 'High', 'High'], fontsize = 14, rotation=45)
axes[0].tick_params(axis='y', which='major', labelsize=14)
#axes[0].set_ylim(0, 1500)

sns.violinplot(het_snp, x='Pop-Consequence', y='Allele Count', order=order, color='0.8', ax=axes[1])
sns.stripplot(het_snp, x='Pop-Consequence', y='Allele Count', hue='Pop-Consequence', hue_order=order, palette=palette, jitter=True, size=4, ax=axes[1])
axes[1].set_title('B)', loc='left', fontsize=20)
axes[1].set_xlabel('')
axes[1].set_ylabel('Count of Heterozygous Derived Alleles', fontsize=16)
axes[1].set_xticklabels(['Low', 'Low', 'Low', 'Moderate', 'Moderate', 'Moderate', 'High', 'High', 'High'], fontsize = 14, rotation=45)
axes[1].tick_params(axis='y', which='major', labelsize=14)
#axes[1].set_ylim(0, 875)

sns.violinplot(alt_snp, x='Pop-Consequence', y='Allele Count', order=order, color='0.8', ax=axes[2])
sns.stripplot(alt_snp, x='Pop-Consequence', y='Allele Count', hue='Pop-Consequence', hue_order=order, palette=palette, jitter=True, size=4, ax=axes[2])
axes[2].set_title('C)', loc='left', fontsize=20)
axes[2].set_xlabel('')
axes[2].set_ylabel('Count of Homozygous Derived Alleles', fontsize=16)
axes[2].set_xticklabels(['Low', 'Low', 'Low', 'Moderate', 'Moderate', 'Moderate', 'High', 'High', 'High'], fontsize = 14, rotation=45)
axes[2].tick_params(axis='y', which='major', labelsize=14)
#axes[2].set_ylim(0, 875)

sns.violinplot(data=combinedRxy, x='Consequence', y='Rxy', color='0.8', ax=axes[3])
sns.stripplot(data=combinedRxy, x='Consequence', y='Rxy', jitter=True, color='#f4811d', dodge=True, ax=axes[3])
axes[3].set_title('D)', loc='left', fontsize=20)
axes[3].set_xlabel('')
axes[3].set_ylabel(r'SNP $R_{XY}$', fontsize=16)
axes[3].set_xticklabels(['Moderate', 'High'], fontsize = 14, rotation=45)
axes[3].tick_params(axis='y', which='major', labelsize=14)

axes[3].axhline(y=1, color='red', linestyle='--', linewidth=0.75)

plt.savefig('Figures/Figure6_load_SNP_summary.png', dpi=300, bbox_inches='tight')

In [ ]:
snp_load = snp_vep[snp_vep['Consequence']!='MODIFIER']
snp_load['Pop-Consequence'] = snp_load['Population'] +'-' + snp_load['Consequence']

tot_snp = snp_load.groupby(['Sample', 'Population', 'Consequence', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
tot_snp = tot_snp[tot_snp['Allele Count'] > 0]

pops = ['AU', 'NZ', 'KI']
titles = ['A)', 'B)', 'C)']

fig = plt.figure(figsize=(15, 8))  # taller figure to fit 3 populations

# Define axes positions manually: [left, bottom, width, height]
# We’ll give each population 0.25 height with a small gap
left, width = 0.1, 0.45
right_left, right_width = 0.57, 0.25
bottoms = [0.7, 0.38, 0.06]  # vertical positions of each population
height = 0.25

consequence_palette = {'LOW':'lightgrey', 'MODERATE':'orange', 'HIGH':'red'}

for i, pop in enumerate(pops):
    # Left (low x-range)
    ax1 = fig.add_axes([left, bottoms[i], width, height])
    data = tot_snp[tot_snp['Population']==pop]
    sns.histplot(
        data,
        x='Allele Count', hue='Consequence', binwidth=20,
        palette=consequence_palette, ax=ax1, legend=False, alpha=0.5
    )
    ax1.set_xlim(0, 1425)
    ax1.set_xlabel('')
    ax1.set_xticks(range(0, 1425, 200))
    ax1.set_ylabel('')
    ax1.set_title(titles[i], loc='left')
    
    # Right (high x-range)
    ax2 = fig.add_axes([right_left, bottoms[i], right_width, height])
    sns.histplot(
        data,
        x='Allele Count', hue='Consequence', binwidth=20,
        palette=consequence_palette, hue_order=['LOW', 'MODERATE', 'HIGH'], ax=ax2, alpha=0.5, legend=(i==0)  # only show legend on first
    )
    ax2.set_xlim(3400, 4000)
    ax2.set_xlabel('')
    ax2.set_xticks(range(3400, 4000, 200))
    ax2.set_ylabel('')  # remove y-label for right axes
    ax2.yaxis.set_ticks([])  # remove y-ticks
    
    # Hide spines between the axes
    ax1.spines.right.set_visible(False)
    ax2.spines.left.set_visible(False)
    ax1.yaxis.tick_left()
    
    # Diagonal break lines
    d = 0.015
    kwargs = dict(marker=[(-1,-1),(1,1)], markersize=8, linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    ax1.plot([1,1], [0,1], transform=ax1.transAxes, **kwargs)
    ax2.plot([0,0], [0,1], transform=ax2.transAxes, **kwargs)

# Common x-label at the bottom
fig.text(0.5, 0.01, 'Total Derived Allele Count', ha='center', fontsize=14)
# Common y-label on the left
fig.text(0.04, 0.5, 'Number of Individuals', va='center', rotation='vertical', fontsize=14)

plt.savefig('Figures/Figure6_load_SNP_summary_derivative.png', dpi=300, bbox_inches='tight')

In [ ]:
high_snp.head(n=25)

In [ ]:
snp_load = snp_vep[snp_vep['Consequence']!='MODIFIER']
snp_load['Pop-Consequence'] = snp_load['Population'] +'-' + snp_load['Consequence']

high_snp = snp_load[snp_load['Consequence']=='HIGH'].groupby(['Sample', 'Population', 'Consequence', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
high_snp = high_snp[high_snp['Allele Count'] > 0]

pops = ['AU', 'NZ', 'KI']
titles = ['A)', 'B)', 'C)']

fig = plt.figure(figsize=(15, 8))  # taller figure to fit 3 populations

palette = {'AU':'gold', 'NZ':'steelblue', 'KI':'black'}
consequence_palette = {'LOW':'lightgrey', 'MODERATE':'orange', 'HIGH':'red'}

for i, pop in enumerate(pops):
    # Left (low x-range)
    ax1 = fig.add_axes([left, bottoms[i], width, height])
    data = high_snp[high_snp['Population']==pop]
    sns.histplot(data, x='Allele Count', hue='Population', palette=palette, ax=ax1, legend=False, alpha=0.5)
    ax1.set_xlim(0, 50)
    ax1.set_xlabel('')
    ax1.set_xticks(range(0, 50, 5))
    ax1.set_ylabel('')
    ax1.set_title(titles[i], loc='left')

#plt.savefig('Figures/Figure6_load_SNP_summary_derivative.png', dpi=300, bbox_inches='tight')


In [ ]:
low_snp = snp_load[snp_load['Consequence']=='LOW'].groupby(['Sample', 'Population', 'Consequence', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
low_snp = low_snp[low_snp['Allele Count'] > 0]
sns.histplot(low_snp, x='Allele Count', hue='Population', palette=palette, kde=True, bins=50, legend=False, alpha=0.5)
#plt.xticks(range(0, 500, 50))
plt.show()

In [ ]:
mod_snp = snp_load[snp_load['Consequence']=='MODERATE'].groupby(['Sample', 'Population', 'Consequence', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
mod_snp = mod_snp[mod_snp['Allele Count'] > 0]
sns.histplot(mod_snp, x='Allele Count', hue='Population', palette=palette, kde=False, bins=50, legend=False, alpha=0.5)
plt.xticks(range(0, 500, 50))
plt.show()

In [ ]:
high_snp = snp_load[snp_load['Consequence']=='HIGH'].groupby(['Sample', 'Population', 'Consequence', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
high_snp = high_snp[high_snp['Allele Count'] > 0]
sns.histplot(high_snp, x='Allele Count', hue='Population', palette=palette, kde=True, bins=50, legend=False, alpha=0.5)
plt.xticks(range(0, 22, 2))
plt.show()

In [ ]:
snp_load = snp_vep[snp_vep['Consequence']!='MODIFIER']
snp_load['Pop-Consequence'] = snp_load['Population'] +'-' + snp_load['Consequence']

alt_snp = snp_load[snp_load['Genotype Category']=='Hom Alt'].groupby(['Sample', 'Population', 'Consequence', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
alt_snp = alt_snp[alt_snp['Allele Count'] > 0]

pops = ['AU', 'NZ', 'KI']
titles = ['A)', 'B)', 'C)']

fig = plt.figure(figsize=(15, 8))  # taller figure to fit 3 populations

# Define axes positions manually: [left, bottom, width, height]
# We’ll give each population 0.25 height with a small gap
left, width = 0.1, 0.45
right_left, right_width = 0.57, 0.25
bottoms = [0.7, 0.38, 0.06]  # vertical positions of each population
height = 0.25

consequence_palette = {'LOW':'lightgrey', 'MODERATE':'orange', 'HIGH':'red'}

for i, pop in enumerate(pops):
    # Left (low x-range)
    ax1 = fig.add_axes([left, bottoms[i], width, height])
    data = alt_snp[alt_snp['Population']==pop]
    sns.histplot(
        data,
        x='Allele Count', hue='Consequence', binwidth=20,
        palette=consequence_palette, ax=ax1, legend=False, alpha=0.5
    )
    ax1.set_xlim(0, 3000)
    ax1.set_xlabel('')
    ax1.set_xticks(range(0, 3000, 200))
    ax1.set_ylabel('')
    ax1.set_title(titles[i], loc='left')


#plt.savefig('Figures/Figure6_load_SNP_summary_derivative.png', dpi=300, bbox_inches='tight')


### SV load plots

In [ ]:
import matplotlib.lines as mlines

indiv_SVdel = sv_vep[sv_vep['Consequence']!='MODIFIER']

indiv_SVdel['Pop-Consequence'] = indiv_SVdel['Population'] + '-' + indiv_SVdel['Consequence']

tot_sv = indiv_SVdel.groupby(['Sample', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
het_sv = indiv_SVdel[indiv_SVdel['Genotype Category']=='Het'].groupby(['Sample', 'Pop-Consequence'])['Allele Count'].sum().reset_index()
alt_sv = indiv_SVdel[indiv_SVdel['Genotype Category']=='Hom Alt'].groupby(['Sample', 'Pop-Consequence'])['Allele Count'].sum().reset_index()

palette = ['gold', 'steelblue', 'black']
order = ['AU-HIGH', 'NZ-HIGH', 'KI-HIGH']

fig, axes = plt.subplots(1, 5, figsize=(30, 8), sharex=False, sharey=False)

sns.violinplot(tot_sv, x='Pop-Consequence', y='Allele Count', order=order, color='0.8', ax=axes[0])
sns.stripplot(tot_sv, x='Pop-Consequence', y='Allele Count', hue='Pop-Consequence', hue_order=order, palette=palette, jitter=True, size=4, ax=axes[0])
axes[0].set_title('E)', loc='left', fontsize=20)
axes[0].set_xlabel('')
axes[0].set_ylabel('Individual Allele Counts', fontsize=16)
axes[0].set_xticklabels(['High', 'High', 'High'], fontsize = 14, rotation=45)
axes[0].tick_params(axis='y', which='major', labelsize=14)
axes[0].set_ylim(0, 200)

sns.violinplot(het_sv, x='Pop-Consequence', y='Allele Count', order=order, color='0.8', ax=axes[1])
sns.stripplot(het_sv, x='Pop-Consequence', y='Allele Count', hue='Pop-Consequence', hue_order=order, palette=palette, jitter=True, size=4, ax=axes[1])
axes[1].set_title('F)', loc='left', fontsize=20)
axes[1].set_xlabel('')
axes[1].set_ylabel('Count of Heterozygous Alleles', fontsize=16)
axes[1].set_xticklabels(['High', 'High', 'High'], fontsize = 14, rotation=45)
axes[1].tick_params(axis='y', which='major', labelsize=14)
axes[1].set_ylim(0, 200)

sns.violinplot(alt_sv, x='Pop-Consequence', y='Allele Count', order=order, color='0.8', ax=axes[2])
sns.stripplot(alt_sv, x='Pop-Consequence', y='Allele Count', hue='Pop-Consequence', hue_order=order, palette=palette, jitter=True, size=4, ax=axes[2])
axes[2].set_title('G)', loc='left', fontsize=20)
axes[2].set_xlabel('')
axes[2].set_ylabel('Count of Homozygous Alleles', fontsize=16)
axes[2].set_xticklabels(['High', 'High', 'High'], fontsize = 14, rotation=45)
axes[2].tick_params(axis='y', which='major', labelsize=14)
axes[2].set_ylim(0, 200)

sns.violinplot(data=svRxy[svRxy['Consequence']!='MODIFIER'], x='Consequence', y='Rxy', color='0.8', ax=axes[3])
sns.stripplot(data=svRxy[svRxy['Consequence']!='MODIFIER'], x='Consequence', y='Rxy', jitter=True, color='#f4811d', dodge=True, ax=axes[3])
axes[3].set_title('H)', loc='left', fontsize=20)
axes[3].set_xlabel('')
axes[3].set_ylabel(r'SV $R_{XY}$', fontsize=16)
axes[3].set_xticklabels(['High'], fontsize = 14, rotation=45)
axes[3].tick_params(axis='y', which='major', labelsize=14)

axes[3].axhline(y=1, color='red', linestyle='--', linewidth=0.75)

axes[4].axis('off')

# Define custom legend elements
legend_labels = ['AFT', 'TI', 'KĪ']
legend_colors = ['gold', 'steelblue', 'black']  # Adjust colors as needed
handles = [mlines.Line2D([], [], color=color, marker='o', markersize=15, linestyle='None', label=label) for color, label in zip(legend_colors, legend_labels)]

# Add the custom legend to the fourth subplot
axes[4].legend(handles=handles, loc='center', fontsize=20, title='Populations', title_fontsize=25)

plt.savefig('Figures/Figure6_load_SV_summary.png', dpi=300, bbox_inches='tight')